# Practical Work 5 - corrected statistical reproduction

This notebook reanalyzes the preserved PR5 results. Latency uses 30 paired measurements. The algorithm comparison collapses the ten identical stored repetitions so that the experimental unit is the dataset (N=5), avoiding pseudoreplication.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, probplot, rankdata, shapiro, studentized_range, wilcoxon

ROOT = Path.cwd()
if not (ROOT / "data" / "pr5").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "pr5"
DATA

## 1. Завантаження 30 експериментальних прогонів

In [ ]:
lat=pd.read_csv(DATA/'pr5_latency_30runs.csv')
lat.describe()

## 2. Тест Шапіро-Уїлка

In [ ]:
for c in ['Baseline (local)','Full validator']:
    s,p=shapiro(lat[c])
    print(c, 'W=',round(s,4),'p=',p)
diff=lat['Full validator']-lat['Baseline (local)']
print('paired differences:', shapiro(diff))

In [ ]:
fig,ax=plt.subplots(figsize=(6,4)); ax.hist(lat['Baseline (local)'],bins=9,edgecolor='black'); ax.set_title('Baseline latency'); ax.set_xlabel('μs/spec'); ax.set_ylabel('frequency'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
probplot(lat['Baseline (local)'], dist='norm', plot=ax)
ax.set_title('Q-Q: Baseline')
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(6,4)); ax.hist(lat['Full validator'],bins=9,edgecolor='black'); ax.set_title('Full validator latency'); ax.set_xlabel('μs/spec'); ax.set_ylabel('frequency'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
probplot(lat['Full validator'], dist='norm', plot=ax)
ax.set_title('Q-Q: Full validator')
plt.show()

## 3. Парне статистичне порівняння

In [ ]:
res_w=wilcoxon(lat['Full validator'],lat['Baseline (local)'])
print(res_w)
print('Median baseline:',lat['Baseline (local)'].median())
print('Median full:',lat['Full validator'].median())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([lat['Baseline (local)'], lat['Full validator']], tick_labels=['Baseline', 'Full'])
ax.set_ylabel('us/spec')
ax.set_title('Latency (30 paired runs)')
plt.show()

## 4. Метрики виявлення невалідних ML-експериментів

In [ ]:
m=pd.read_csv(DATA/'pr5_classification_metrics.csv'); m

ROC-AUC обчислено за числом виявлених порушень як скоринговою функцією. Для повного валідатора на контрольованому benchmark усі 8 типів порушень мають відповідне правило, тому AUC=1.0.

## 5. Three algorithms x five independent benchmark datasets

The source table contains ten rows per dataset/algorithm, but their F1 and accuracy values are identical. They are collapsed to one observation per dataset before inferential testing.

In [ ]:
raw = pd.read_csv(DATA / 'pr5_three_algorithms_5datasets.csv')
variation = raw.groupby(['dataset', 'algorithm'])[['f1', 'accuracy']].nunique()
assert (variation == 1).all().all(), 'Stored repetitions unexpectedly vary; review the analysis unit.'

r = raw.groupby(['dataset', 'algorithm'], as_index=False)[['f1', 'accuracy']].mean()
p = r.pivot(index='dataset', columns='algorithm', values='f1')
print('Independent blocks:', len(p))
p

In [ ]:
alg=['Baseline (local)','Full validator','Ablation - provenance']
fr=friedmanchisquare(*[p[a] for a in alg]); print('Friedman:',fr)
ranks=np.vstack([rankdata(-row[alg].to_numpy(),method='average') for _,row in p.iterrows()])
avg=ranks.mean(axis=0); print(dict(zip(alg,avg)))

## 6. Nemenyi post-hoc using N=5 independent dataset blocks

In [ ]:
import math
N=len(p); k=len(alg); se=math.sqrt(k*(k+1)/(6*N))
for i in range(k):
    for j in range(i+1,k):
        q=abs(avg[i]-avg[j])/se
        pv=studentized_range.sf(q*math.sqrt(2),k,np.inf)
        print(alg[i], 'vs', alg[j], 'p=',pv)

## 7. Cohen's dz

In [ ]:
def dz(a,b):
    d=np.asarray(a)-np.asarray(b); return d.mean()/d.std(ddof=1)
for i in range(k):
    for j in range(i+1,k):
        print(alg[i], 'vs', alg[j], dz(p[alg[i]],p[alg[j]]))

## Conclusion

On this controlled synthetic benchmark, the full validator has the highest recall and F1 and a statistically significant latency increase relative to the local baseline. After correcting the experimental unit from 50 duplicated rows to five independent datasets, the Friedman test remains significant, but the Nemenyi comparisons involving the provenance ablation are not significant at alpha=0.05. These results demonstrate rule coverage on the controlled benchmark; they do not establish generalization to real ML projects.